# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID: {metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each entity (record set, field, column) is referenced by its `@id`. Here, we'll print all record sets and their contained fields by their IDs.

In [ ]:
# List all record sets and their fields (by @id)
record_sets = [r['@id'] for r in dataset.metadata.record_sets]
print("Available record sets (by @id):")
for rset in dataset.metadata.record_sets:
    print(f"- {rset['@id']} (name: {rset.get('name', '')})")
    fields = [f['@id'] for f in rset.get('fields', [])]
    print(f"  Fields in this record set:")
    for f in rset.get('fields', []):
        print(f"    * Field @id: {f['@id']}, label: {f.get('label', f.get('name', ''))}")

Now let's preview the contents of the first record set by its `@id`.

In [ ]:
# Preview records from the first record set
first_record_set_id = record_sets[0]
print(f"Sample records from record set @id: {first_record_set_id}")
for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
    print(record)
    if i == 2:
        break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
dataframes = {}
for rset_id in record_sets:
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df

# Print columns of the first record set
print(f"Columns in record set {first_record_set_id}:")
print(dataframes[first_record_set_id].columns.tolist())
dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations like removing outliers, transforming data distributions, or grouping data by key attributes are useful to prepare data for further analysis.

**Example:** Filter GAD-7 scores above a threshold, normalize, and group by gender.

In [ ]:
# Identify a numeric field in the record set (by @id).
# Suppose the @id of GAD-7 score field is 'http://mlcommons.org/croissant/field/gad7_score' and gender field is 'http://mlcommons.org/croissant/field/gender'

numeric_field_id = 'http://mlcommons.org/croissant/field/gad7_score'
group_field_id = 'http://mlcommons.org/croissant/field/gender'

# Choose the first record set for demonstration
record_set_id = first_record_set_id

# Confirm field exists
df = dataframes[record_set_id]
if numeric_field_id in df.columns:
    # Filter records with GAD-7 score above threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by gender
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df)
else:
    print(f"Field {numeric_field_id} not found in columns: {df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot GAD-7 score distribution and compare by gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize GAD-7 score distribution
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title('Distribution of GAD-7 Scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Frequency')
    plt.show()

# Compare mean scores by gender if available
if numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title('GAD-7 Scores by Gender')
    plt.xlabel('Gender')
    plt.ylabel('GAD-7 Score')
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the Croissant metadata and `mlcroissant` Python library. Key steps included referencing entities by their `@id` fields, extracting data from multiple record sets, performing EDA, and visualizing key results such as GAD-7 score distributions.

**Next steps:** Extend this notebook to explore other survey scores (PHQ-9, PSQ), apply advanced statistical methods, or integrate additional demographic attributes for deeper insights.